In [4]:
import math
import os
import sys
sys.path.append(os.path.abspath('..')) # to run files that are away
os.environ["WANDB_SILENT"] = "true"  # Suppress WandB logs

libraries = ["torch", "numpy", "polars"]
modules   = {lib: sys.modules.get(lib) for lib in libraries}

if not modules["torch"]:
    import torch
if not modules["numpy"]:
    import numpy as np
if not modules["polars"]:
    import polars as pl

import gc
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
# from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from utils import Losses, DimensionalityEstimator
from glob import glob
import polars as pl


device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
"""Load WAFER data files"""
# Split wafer_df by wafer, then reshape each wafer df to produce y_df

main_folder = "../ASM_data"
dict_of_wafer_files = {'file1': {'path': f"{main_folder}/2. marathon0/Wafer performance/Spatial property after step 4.csv", 'marathon': 0},
                       'file2': {'path': f"{main_folder}/3. marathon1/Wafer performance/Spatial property.csv", 'marathon': 1}}

wafer_master_lf = pl.concat([
    pl.scan_csv(e['path'], ignore_errors=True)
      .with_columns([
          pl.lit(e['marathon']).alias("marathon"),
          (pl.lit(e['marathon']).cast(pl.Utf8) + "_" + pl.col("#Run").cast(pl.Utf8)).alias("marathon_run")])
    for e in dict_of_wafer_files.values()], how="diagonal")

cols = wafer_master_lf.columns
for c in ["marathon", "marathon_run"]:
    if c in cols: # Check if column exists before removing
        cols.remove(c)
insert_idx = cols.index("#Run") + 1
cols[insert_idx:insert_idx] = ["marathon", "marathon_run"]

# Select columns in desired order (still lazy)
wafer_master_lf = wafer_master_lf.select(cols)

print(f"WAFER df initial schema (lazy): {wafer_master_lf.schema}")

# Save to parquet (this will trigger computation)
# Keeping this commented as per your original code
# should_we_save = False
# if should_we_save:
#     saving_location = f"{main_folder}/parquet_files/master_wafer_file.parquet"
#     wafer_master_lf.sink_parquet(saving_location) # Use sink_parquet for lazy writing
#     print(f"\nConsolidated data saved to: {saving_location}")

# No need to clone `wafer_master_df` if you are immediately partitioning.
# `wafer_master_df` will be the result of the `collect()` call.

def reshape_wafer_df_to_y_and_radius(wafer_df_partition):
    # Perform pivot and aggregation lazily if possible, then collect
    # If the partition is already small enough, collect() here is fine
    # Ensure intermediate results are not kept unnecessarily
    
    pivot_df = wafer_df_partition.pivot(values="Radius (mm)",
                                        index=["marathon_run", "RC"],
                                        columns="Site #",
                                        aggregate_function="mean")
    pivot_df = pivot_df.rename({str(c): f"site_{c}" for c in pivot_df.columns if isinstance(c, int) or c.isdigit()})

    # Then aggregate over RC to get one row per marathon_run
    final_wafer_df = (pivot_df.group_by("marathon_run").agg(pl.all().mean()))
    final_wafer_df = final_wafer_df.sort("marathon_run")

    y_df = wafer_df_partition.pivot(values="Spatial property (nm)",
                                    index="marathon_run",
                                    columns="Site #",
                                    aggregate_function="mean")
    y_df = y_df.sort("marathon_run")
    
    # Selecting only necessary columns for radius_df
    radius_df = wafer_df_partition.select(["marathon_run", "Radius (mm)"])
    return y_df, radius_df

# Collect once to partition. If this still causes memory issues,
# you'd need to process each 'RC' group from the raw files individually.
wafer_master_df = wafer_master_lf.collect() # This will materialize the DataFrame

# Partitioning and processing in a loop
y_df_dict = {}
radius_df_dict = {}
radius_wide_dict = {}

# Process partitions one by one to limit memory
for i, (rc_value, df_partition) in enumerate(wafer_master_df.group_by("RC")):
    # No need to add 'wafer' column here, as it's added later when joining
    # This also saves memory by not adding redundant columns to individual partitions
    y_df_dict[i], current_radius_df = reshape_wafer_df_to_y_and_radius(df_partition)

    radius_df = (current_radius_df.sort("marathon_run")
                                  .with_columns(pl.arange(0, pl.count()).over("marathon_run").alias("radius_idx")))
    radius_wide = radius_df.pivot(values="Radius (mm)",
                                  index="marathon_run",
                                  columns="radius_idx",
                                  aggregate_function="first").sort("marathon_run")
    radius_wide_dict[i] = radius_wide

    del current_radius_df, radius_df, radius_wide, df_partition # Delete processed DataFrames
    gc.collect()

del wafer_master_df, cols, insert_idx # Clean up top-level wafer DFs and helper vars
gc.collect()

print("WAFER data processing complete. Intermediate wafer DFs cleaned.")

WAFER df initial schema (lazy): Schema([('#Run', Int64), ('marathon', Int32), ('marathon_run', String), ('Site #', Int64), ('RC', Int64), ('Spatial property (nm)', Float64), ('Radius (mm)', Int64)])
WAFER data processing complete. Intermediate wafer DFs cleaned.


In [7]:
"""Load LOG data files"""

main_folder = "../ASM_data"
dict_of_log_files = {'file1': {'path': f"{main_folder}/2. marathon0/logs/Step1.csv", 'step': 1, 'marathon': 0},
                     'file2': {'path': f"{main_folder}/2. marathon0/logs/Step2.csv", 'step': 2, 'marathon': 0},
                     'file3': {'path': f"{main_folder}/2. marathon0/logs/Step3.csv", 'step': 3, 'marathon': 0},
                     'file4': {'path': f"{main_folder}/2. marathon0/logs/Step4.csv", 'step': 4, 'marathon': 0},
                     'file5': {'path': f"{main_folder}/3. marathon1/logs/Step1.csv", 'step': 1, 'marathon': 1},
                     'file6': {'path': f"{main_folder}/3. marathon1/logs/Step2.csv", 'step': 2, 'marathon': 1},
                     'file7': {'path': f"{main_folder}/3. marathon1/logs/Step3.csv", 'step': 3, 'marathon': 1},
                     'file8': {'path': f"{main_folder}/3. marathon1/logs/Step4.csv", 'step': 4, 'marathon': 1},}
step_col_name = "step_id"
df_list       = []
COMMON_ID_COLS= ['Process Time', '#Run']
COMMON_ID_COLS_MOD = [c for c in COMMON_ID_COLS if c != "#Run"] + ["marathon_run"]

def process_log_file(entry, step_col_name, common_id_cols_mod):
    file_path = entry['path']
    step_id = entry['step']
    marathon = entry['marathon']
    print(f"Processing {file_path} for step {step_id} and marathon {marathon}...")

    # Use scan_csv for lazy loading. It's crucial that '#Run' exists in the raw CSV.
    df = pl.scan_csv(file_path, ignore_errors=True)

    # 1. First, create 'marathon_run' using the original '#Run' column name.
    #    Add a check to ensure '#Run' exists before trying to cast it.
    #    If not all files have '#Run', you might need to handle it gracefully (e.g., fill with nulls or skip).
    #    Assuming '#Run' is always present based on your original code's implicit assumption.
    if "#Run" not in df.columns: # Check if the column exists in the lazy frame's projected schema
        # This branch might be needed if some files genuinely lack #Run
        # For this specific error, it implies it exists but the rename is the issue.
        raise pl.ColumnNotFoundError(f"'#Run' not found in {file_path}. Please check CSV files.")

    df = df.with_columns([
        pl.lit(marathon).alias("marathon"),
        pl.lit(step_id).alias(step_col_name),
        (pl.col("marathon").cast(pl.Utf8) + "_" + pl.col("#Run").cast(pl.Utf8)).alias("marathon_run")
    ])

    # 2. Then, perform the general column renaming.
    #    This way, 'marathon_run' is created based on '#Run' before '#Run' itself is potentially renamed.
    #    The original rename function used `df.columns` which would collect the schema.
    #    For lazy frames, you need to be careful. Let's make it robust by checking if `c` is in `df.columns`
    #    or, more accurately, by performing this rename on the full, collected DF if column names
    #    can vary significantly or contain special chars that `strip().title()` doesn't handle well with lazy.
    
    # For robust lazy renaming, it's safer to map columns found in the schema before the .collect()
    # or ensure your `rename_mapping` function properly generates for the lazy context.
    # The original rename({c: c.strip().title() ...}) might cause issues if a column is missing after title()
    # Let's assume the strip().title() logic is sound for all columns you want.
    
    # It's safer to generate the rename mapping based on the columns *before* title casing.
    # Or, if the issue is `#Run` -> `Run`, then you need to be precise.
    # Let's adjust it to ensure `#Run` is handled.

    # Option A: Handle '#Run' specifically and then do general renaming
    rename_dict = {}
    for col in df.columns:
        stripped_titled_col = col.strip().title()
        if col == '#Run': # Keep original for creating 'marathon_run', but rename after
            rename_dict[col] = 'Run' # Or whatever you prefer, e.g., 'Run_Original'
        else:
            rename_dict[col] = stripped_titled_col
            
    df = df.rename(rename_dict) # Apply the renaming

    # Now, cast numeric columns. The schema lookup with lazy frames is usually fine here.
    # Ensure this part is robust.
    df = df.with_columns([
        pl.col(col).cast(pl.Float64)
        for col in df.columns # Use updated columns after renaming
        if df.schema[col] in [pl.Int64, pl.Float32, pl.Int32, pl.UInt64, pl.UInt32, pl.Int16, pl.UInt16, pl.Int8, pl.UInt8]
    ])

    # The rename_mapping for adding step_id suffix should use the *new* column names.
    # This part should be correct if `df.columns` reflects the `strip().title()` renaming.
    suffix_rename_mapping = {col: f"{col}_step{step_id}"
                             for col in df.columns
                             if col not in common_id_cols_mod + [step_col_name, "marathon"]} # "marathon" also should not be suffixed
    df = df.rename(suffix_rename_mapping)


    # Drop single-value columns will be done on the collected master_df.
    # No changes needed here, as it's commented out in your original `process_log_file`.

    return df # Return a LazyFrame
# Use list comprehension with scan_csv for lazy processing
df_list_lazy = [process_log_file(entry, step_col_name, COMMON_ID_COLS_MOD)
                for entry in dict_of_log_files.values()]

# Concatenate lazy frames
master_lf = pl.concat(df_list_lazy, how="diagonal")

# --- Materialize master_df once for further processing ---
# This is a critical point for memory usage. If this crashes,
# you'll need to re-evaluate the overall strategy to avoid a single large DF.
# For now, let's assume this can be done.
master_df = master_lf.collect()
print(f"Master log df shape after initial concat and collect: {master_df.shape}")

# Now perform operations that require materialized data
# Drop single-value columns (now with actual data)
single_val_cols = [col for col in master_df.columns
                   if col not in COMMON_ID_COLS_MOD + [step_col_name]
                   and master_df[col].n_unique() == 1]
master_df = master_df.drop(single_val_cols)

# Fill nulls with 0 in numeric cols except COMMON_ID_COLS_MOD
for col in master_df.columns:
    if master_df[col].dtype.is_numeric() and col not in COMMON_ID_COLS_MOD:
        master_df = master_df.with_columns(pl.col(col).fill_null(0))

def reorder_cols(df, step_col_name):
    """Reorder columns: COMMON_ID_COLS_MOD + step_col_name + rest"""
    desired_order = COMMON_ID_COLS_MOD + [step_col_name] + [
        col for col in df.columns if col not in COMMON_ID_COLS_MOD + [step_col_name]]
    # Ensure all desired columns exist before selecting
    desired_order = [col for col in desired_order if col in df.columns]
    return df.select(desired_order)

master_df = reorder_cols(master_df, step_col_name)
log_df = master_df.clone() # Keep log_df for later use, if needed

print(f"Filtered Master df shape: {master_df.shape}")

# Save to parquet (master log file)
# should_we_save = False
# if should_we_save:
#     saving_location = f"{main_folder}/parquet_files/master_log_file.parquet"
#     master_df.write_parquet(saving_location)
#     print(f"\nConsolidated data saved to: {saving_location}")

del df_list_lazy, master_lf, master_df # Clean up intermediate lazy frames and master_df
gc.collect()

# --- Split log_df into 4, 1 for each wafer and save to parquet immediately ---
# This is where your code was doing well in terms of memory.
log_df2 = log_df.rename({c: c.strip().lower() for c in log_df.columns})
del log_df # Free up memory from the original log_df

common_cols = [c for c in log_df2.columns if not c.startswith("rc")]

for i in range(4):
    wafer_cols = [c for c in log_df2.columns if c.startswith(f"rc{i+1}")]
    cols_to_keep = common_cols + wafer_cols
    
    # Select and add 'wafer' column. No need for clone().
    df_with_wafer = log_df2.select(cols_to_keep).with_columns(pl.lit(i+1).alias("wafer"))

    # Reorder 'wafer' after 'step_id'
    if "step_id" in (cols := df_with_wafer.columns):
        cols.remove("wafer")
        insert_idx = cols.index("step_id") + 1
        cols = cols[:insert_idx] + ["wafer"] + cols[insert_idx:]
        df_with_wafer = df_with_wafer.select(cols)

    # Drop constant-valued numeric columns (now with the specific wafer data)
    constant_cols = [col for col in df_with_wafer.columns
                     if df_with_wafer[col].dtype.is_numeric() and df_with_wafer[col].n_unique() == 1]
    df_with_wafer = df_with_wafer.drop(constant_cols)
    
    # Write to parquet immediately and delete
    df_with_wafer.write_parquet(f"{main_folder}/parquet_files/wafer_{i+1}_log.parquet")
    del df_with_wafer # Explicitly delete
    gc.collect()

del log_df2, common_cols # Clean up
gc.collect()

Processing ../ASM_data/2. marathon0/logs/Step1.csv for step 1 and marathon 0...


ColumnNotFoundError: marathon

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'with_columns' <---
Csv SCAN [../ASM_data/2. marathon0/logs/Step1.csv]
PROJECT */183 COLUMNS

In [ ]:
# Assuming the reshape_log_df and summary_log_df_dict creation
# is part of the original logic, we need to adapt it to read from parquet
# instead of keeping everything in memory.

from lightgbm import LGBMRegressor, early_stopping
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

def join_logs_and_wafer_df(log_df, wafer_df, y_df):
    """to get X"""
    X_full    = log_df.join(wafer_df, on="marathon_run", how="inner", suffix="_df2")
    X_full_pd = X_full.to_pandas()
    y_full_pd = y_df.sort("marathon_run").drop("marathon_run").to_pandas()
    return X_full_pd, y_full_pd

def scale_and_split_data(X_full_pd, y_full_pd):
    y_scaler      = StandardScaler()
    y_full_scaled = y_scaler.fit_transform(y_full_pd)

    X_train, X_val, y_train, y_val = train_test_split(X_full_pd, y_full_scaled, test_size=0.2, random_state=42)

    cols_to_scale = [c for c in X_train.select_dtypes(include=np.number).columns if c != "marathon_run"]
    scaler        = StandardScaler()

    X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_val[cols_to_scale]   = scaler.transform(X_val[cols_to_scale])

    X_train_final = X_train.drop(columns=["marathon_run"])
    X_val_final   = X_val.drop(columns=["marathon_run"])

    return X_train_final, y_train, X_val_final, y_val, y_scaler

def predict_multioutput_lightgbm(X_train, y_train, X_val, y_val):
    n_targets = y_train.shape[1]
    y_pred    = np.zeros(y_val.shape)
    
    for i in range(n_targets):
        model = LGBMRegressor(objective='regression',
                              verbosity=-1,
                              n_estimators=1000,
                              learning_rate=0.05,
                              num_leaves=31,
                              max_depth=-1,
                              min_data_in_leaf=20)
        model.fit(X_train, y_train[:, i],
                  eval_set=[(X_val, y_val[:, i])],
                  callbacks=[early_stopping(stopping_rounds=50, verbose=False)])
        y_pred[:, i] = model.predict(X_val)
    
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    return rmse, y_pred

def predict_multioutput_catboost(X_train, y_train, X_val, y_val):
    model = MultiOutputRegressor(cb.CatBoostRegressor(verbose=0,
                                                      iterations=100,
                                                      task_type=device))
    model.fit(X_train, y_train)
    y_pred_cat = model.predict(X_val)
    rmse_cat = mean_squared_error(y_val, y_pred_cat) ** 0.5
    return rmse_cat, y_pred_cat

def predict_multioutput_xgboost(X_train, y_train, X_val, y_val):
    model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', verbosity=0))
    model.fit(X_train, y_train)
    y_pred_xgb = model.predict(X_val)
    rmse_xgb = mean_squared_error(y_val, y_pred_xgb) ** 0.5
    return rmse_xgb, y_pred_xgb

def predict_multioutput_randomforest(X_train, y_train, X_val, y_val):
    rf_model  = MultiOutputRegressor(RandomForestRegressor(random_state=42))
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_val)
    rmse_rf   = mean_squared_error(y_val, y_pred_rf) ** 0.5
    return rmse_rf, y_pred_rf

def predict_multioutput_hgb(X_train, y_train, X_val, y_val):
    model = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=100))
    model.fit(X_train, y_train)
    y_pred_hgb = model.predict(X_val)
    rmse_hgb = mean_squared_error(y_val, y_pred_hgb) ** 0.5
    return rmse_hgb, y_pred_hgb

def predict_multioutput_elasticnet(X_train, y_train, X_val, y_val):
    imputer = SimpleImputer(strategy="mean")
    base_model = make_pipeline(imputer, ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=1000))
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)
    y_pred_elas = model.predict(X_val)
    rmse_elas = mean_squared_error(y_val, y_pred_elas) ** 0.5
    return rmse_elas, y_pred_elas

def summarize_log_df(log_df_path, col_to_group_by, y_df):
    """
    Reads log_df from parquet, summarizes, and filters.
    Operates on a LazyFrame to minimize memory.
    """
    log_lf = pl.scan_parquet(log_df_path) # Start with lazy frame from parquet

    stat_map = {"mean": lambda col: pl.col(col).mean(),
                "std": lambda col: pl.col(col).std(),
                "min": lambda col: pl.col(col).min(),
                "max": lambda col: pl.col(col).max(),
                "median": lambda col: pl.col(col).median(),
                "skew": lambda col: pl.col(col).skew(),
                "kurt": lambda col: pl.col(col).kurtosis()}

    exclude_substrings = ['process time', '#run', 'wafer', 'step_id', 'marathon_run']

    # Identify numeric columns for aggregation. This can be tricky with lazy frames
    # if you don't know the schema beforehand. A safe approach is to perform the
    # aggregation on all numeric-like columns and then filter out undesired ones later.
    # For now, let's assume the schema is consistent and `log_lf.schema` can provide types.

    # To get numeric columns from a lazy frame, you might need to infer or hardcode.
    # A more robust way might be to peek at the schema of a small sample or the Parquet file.
    # For now, let's modify it to be less reliant on `df.schema` in the loop directly.

    # Determine numeric columns from the schema of the actual parquet file
    # Or, if you know the typical structure, hardcode common numeric prefixes/suffixes.
    
    # Temporarily read a small part to get schema, or assume column naming conventions.
    # For large files, it's better to avoid materializing even a sample if not strictly necessary.
    # Let's assume most columns are numeric and exclude known non-numeric ones.
    
    # A better way for lazy frames:
    # Get column names after all transformations applied before `collect()`
    # You might need to adjust `exclude_substrings` based on actual column names after `rename_mapping`
    
    numeric_cols = [c for c in log_lf.columns
                    if not any(sub in c.lower() for sub in exclude_substrings)
                    and log_lf.schema[c] in {pl.Float64, pl.Float32, pl.Int64, pl.Int32,
                                             pl.UInt64, pl.UInt32, pl.Int16, pl.UInt16, pl.Int8, pl.UInt8}]

    agg_exprs = []
    for stat_name, func in stat_map.items():
        for col in numeric_cols:
            # Ensure the alias doesn't create duplicate names if columns were already renamed
            agg_exprs.append(func(col).alias(f"{col}_{stat_name}"))

    result_lf = log_lf.group_by(col_to_group_by).agg(agg_exprs)
    
    # Filter only for marathon_run values present in y_df (which is already collected)
    filtered_log_lf = result_lf.filter(pl.col(col_to_group_by).is_in(y_df[col_to_group_by].to_list()))
    
    return filtered_log_lf.collect() # Collect here to get a DataFrame for joining

# Model training loop
rmse_lgb_total = 0
rmse_cat_total = 0
count = 0

for i in range(4):
    print(f"Processing Wafer {i+1} for model training...")
    
    # Read wafer-specific log data from parquet, process, and collect
    # This ensures only one wafer's log data is in memory at a time.
    current_wafer_log_df = summarize_log_df(f"{main_folder}/parquet_files/wafer_{i+1}_log.parquet", 'marathon_run', y_df_dict[i])
    current_wafer_log_df = current_wafer_log_df.with_columns(pl.lit(i+1).alias("wafer"))

    # Join with radius data (already in memory from earlier processing)
    # Ensure 'marathon_run' is common and correctly typed for join
    current_wafer_log_df = current_wafer_log_df.join(radius_wide_dict[i], on="marathon_run", how="left")
    
    # Reorder columns, making 'wafer' the first column (optional, but good for consistency)
    cols = current_wafer_log_df.columns
    current_wafer_log_df = current_wafer_log_df.select(["wafer"] + [c for c in cols if c != "wafer"])

    # Convert to Pandas for model training
    # Only convert necessary columns for X_full_pd to reduce memory
    X_full_pd = current_wafer_log_df.to_pandas().drop(columns=["wafer"], errors='ignore')
    
    # Ensure y_full_pd is a Pandas DataFrame for consistency with scikit-learn
    y_full_pd = y_df_dict[i].sort("marathon_run").drop("marathon_run").to_pandas()
    
    # Free up Polars DFs no longer needed
    del current_wafer_log_df
    gc.collect()

    X_train_final, y_train, X_val_final, y_val, y_scaler = scale_and_split_data(X_full_pd, y_full_pd)

    # Delete large Pandas DFs after scaling and splitting
    del X_full_pd, y_full_pd
    gc.collect()

    print(f"Wafer {i+1}")
    rmse_lgb, y_pred_lgb = predict_multioutput_lightgbm(X_train_final, y_train, X_val_final, y_val)
    print(f"LGBM RMSE: {rmse_lgb}")

    rmse_cat, y_pred_cat = predict_multioutput_catboost(X_train_final, y_train, X_val_final, y_val)
    print(f"Catboost RMSE: {rmse_cat}")

    rmse_lgb_total += rmse_lgb
    rmse_cat_total += rmse_cat
    count += 1
    
    # Delete model-specific variables and data after each iteration
    del X_train_final, y_train, X_val_final, y_val, y_scaler, y_pred_lgb, y_pred_cat
    gc.collect() # Force garbage collection

print(f"\nAverage LGBM RMSE: {rmse_lgb_total / count}")
print(f"Average Catboost RMSE: {rmse_cat_total / count}")

# Final cleanup of remaining dictionaries
del y_df_dict, radius_wide_dict
gc.collect()

In [11]:
from lightgbm import LGBMRegressor, early_stopping
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

def join_logs_and_wafer_df(log_df, wafer_df, y_df):
    """to get X"""
    X_full    = log_df.join(wafer_df, on="marathon_run", how="inner", suffix="_df2")
    X_full_pd = X_full.to_pandas()
    y_full_pd = y_df.sort("marathon_run").drop("marathon_run").to_pandas()
    return X_full_pd, y_full_pd

def scale_and_split_data(X_full_pd, y_full_pd):
    y_scaler      = StandardScaler()
    y_full_scaled = y_scaler.fit_transform(y_full_pd)

    X_train, X_val, y_train, y_val = train_test_split(X_full_pd, y_full_scaled, test_size=0.2, random_state=42)

    cols_to_scale = [c for c in X_train.select_dtypes(include=np.number).columns if c != "marathon_run"]
    scaler        = StandardScaler()

    X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_val[cols_to_scale]   = scaler.transform(X_val[cols_to_scale])

    X_train_final = X_train.drop(columns=["marathon_run"])
    X_val_final   = X_val.drop(columns=["marathon_run"])

    return X_train_final, y_train, X_val_final, y_val, y_scaler

def predict_multioutput_lightgbm(X_train, y_train, X_val, y_val):
    n_targets = y_train.shape[1]
    y_pred    = np.zeros(y_val.shape)
    
    for i in range(n_targets):
        model = LGBMRegressor(objective='regression',
                              verbosity=-1,
                              n_estimators=1000,
                              learning_rate=0.05,
                              num_leaves=31,
                              max_depth=-1,
                              min_data_in_leaf=20)
        model.fit(X_train, y_train[:, i],
                  eval_set=[(X_val, y_val[:, i])],
                  callbacks=[early_stopping(stopping_rounds=50, verbose=False)])
        y_pred[:, i] = model.predict(X_val)
    
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    return rmse, y_pred

def predict_multioutput_catboost(X_train, y_train, X_val, y_val):
    model = MultiOutputRegressor(cb.CatBoostRegressor(verbose=0,
                                                      iterations=100,
                                                      task_type=device))
    model.fit(X_train, y_train)
    y_pred_cat = model.predict(X_val)
    rmse_cat = mean_squared_error(y_val, y_pred_cat) ** 0.5
    return rmse_cat, y_pred_cat

def predict_multioutput_xgboost(X_train, y_train, X_val, y_val):
    model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', verbosity=0))
    model.fit(X_train, y_train)
    y_pred_xgb = model.predict(X_val)
    rmse_xgb = mean_squared_error(y_val, y_pred_xgb) ** 0.5
    return rmse_xgb, y_pred_xgb

def predict_multioutput_randomforest(X_train, y_train, X_val, y_val):
    rf_model  = MultiOutputRegressor(RandomForestRegressor(random_state=42))
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_val)
    rmse_rf   = mean_squared_error(y_val, y_pred_rf) ** 0.5
    return rmse_rf, y_pred_rf

def predict_multioutput_hgb(X_train, y_train, X_val, y_val):
    model = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=100))
    model.fit(X_train, y_train)
    y_pred_hgb = model.predict(X_val)
    rmse_hgb = mean_squared_error(y_val, y_pred_hgb) ** 0.5
    return rmse_hgb, y_pred_hgb

def predict_multioutput_elasticnet(X_train, y_train, X_val, y_val):
    imputer = SimpleImputer(strategy="mean")
    base_model = make_pipeline(imputer, ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=1000))
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)
    y_pred_elas = model.predict(X_val)
    rmse_elas = mean_squared_error(y_val, y_pred_elas) ** 0.5
    return rmse_elas, y_pred_elas

# ==================
# all-in-1 df

# X_full_pd, y_full_pd = join_logs_and_wafer_df(filtered_log_df, final_wafer_df, final_y_df)
# X_train_final, y_train, X_val_final, y_val, _ = scale_and_split_data(X_full_pd, y_full_pd)

# rmse_lgb, y_pred_lgb = predict_multioutput_lightgbm(X_train_final, y_train, X_val_final, y_val)
# print(f"LGBM RMSE: {rmse_lgb}")

# rmse_cat, y_pred_cat = predict_multioutput_catboost(X_train_final, y_train, X_val_final, y_val)
# print(f"Catboose RMSE: {rmse_cat}")

# rmse_rf, y_pred_rf = predict_multioutput_randomforest(X_train_final, y_train, X_val_final, y_val)
# print(f"Random Forest RMSE: {rmse_rf}")

# rmse_xgb, y_pred_xgb = predict_multioutput_xgboost(X_train_final, y_train, X_val_final, y_val)
# print(f"XGBoost RMSE: {rmse_xgb}")

# rmse_hgb, y_pred_hgb = predict_multioutput_hgb(X_train_final, y_train, X_val_final, y_val)
# print(f"HistGB RMSE: {rmse_hgb}")

# rmse_elas, y_pred_elas = predict_multioutput_elasticnet(X_train_final, y_train, X_val_final, y_val)
# print(f"Elastic RMSE: {rmse_elas}")

# =============
# loop through 4 df. we joined X previously

rmse_lgb_total = 0
rmse_cat_total = 0
# rmse_rf_total  = 0
# rmse_xgb_total = 0
# rmse_hgb_total = 0
# rmse_elas_total= 0

count = 0
# ===============
for i in range(4):
    wafer_log_df = pl.read_parquet(f"{main_folder}/parquet_files/wafer_{i+1}_log.parquet")

    summary_df = summarize_log_df(wafer_log_df, 'marathon_run', y_df_dict[i])
    summary_df = summary_df.with_columns(pl.lit(i+1).alias("wafer"))
    summary_df = summary_df.join(radius_wide_dict[i], on="marathon_run", how="left")
    cols = summary_df.columns
    summary_df = summary_df.select(["wafer"] + [c for c in cols if c != "wafer"])

    X_full_pd = summary_df.to_pandas().drop(columns=["wafer"], errors='ignore')
    y_full_pd = y_df_dict[i].sort("marathon_run").drop("marathon_run").to_pandas()
    X_train_final, y_train, X_val_final, y_val, y_scaler = scale_and_split_data(X_full_pd, y_full_pd)

    print(f"Wafer {i+1}")
    rmse_lgb, y_pred_lgb = predict_multioutput_lightgbm(X_train_final, y_train, X_val_final, y_val)
    print(f"LGBM RMSE: {rmse_lgb}")

    rmse_cat, y_pred_cat = predict_multioutput_catboost(X_train_final, y_train, X_val_final, y_val)
    print(f"Catboost RMSE: {rmse_cat}")

    del wafer_log_df, summary_df, X_full_pd, y_full_pd, X_train_final, y_train, X_val_final, y_val
    gc.collect()
# ===============
# for i, X_full in summary_log_df_dict.items():
#     X_full_pd = X_full.to_pandas().drop(columns=["wafer"], errors='ignore')
#     y_full_pd = y_df_dict[i].sort("marathon_run").drop("marathon_run").to_pandas()
#     X_train_final, y_train, X_val_final, y_val, y_scaler = scale_and_split_data(X_full_pd, y_full_pd)

#     print(f"Wafer {i+1}")
#     rmse_lgb, y_pred_lgb = predict_multioutput_lightgbm(X_train_final, y_train, X_val_final, y_val)
#     print(f"LGBM RMSE: {rmse_lgb}")

#     rmse_cat, y_pred_cat = predict_multioutput_catboost(X_train_final, y_train, X_val_final, y_val)
#     print(f"Catboost RMSE: {rmse_cat}")

#     # rmse_rf, y_pred_rf = predict_multioutput_randomforest(X_train_final, y_train, X_val_final, y_val)
#     # print(f"Random Forest RMSE: {rmse_rf}")

#     # rmse_xgb, y_pred_xgb = predict_multioutput_xgboost(X_train_final, y_train, X_val_final, y_val)
#     # print(f"XGBoost RMSE: {rmse_xgb}")

#     # rmse_hgb, y_pred_hgb = predict_multioutput_hgb(X_train_final, y_train, X_val_final, y_val)
#     # print(f"HistGB RMSE: {rmse_hgb}")

#     # rmse_elas, y_pred_elas = predict_multioutput_elasticnet(X_train_final, y_train, X_val_final, y_val)
#     # print(f"Elastic RMSE: {rmse_elas}")

#     rmse_lgb_total += rmse_lgb
#     rmse_cat_total += rmse_cat
#     # rmse_rf_total  += rmse_rf
#     # rmse_xgb_total += rmse_xgb
#     # rmse_hgb_total += rmse_hgb
#     # rmse_elas_total+= rmse_elas
#     count += 1
# ===============

print(f"Average LGBM RMSE: {rmse_lgb_total / count}")
print(f"Average Catboost RMSE: {rmse_cat_total / count}")
# print(f"Average rf RMSE: {rmse_rf_total / count}")
# print(f"Average xgb RMSE: {rmse_xgb_total / count}")
# print(f"Average hgb RMSE: {rmse_hgb_total / count}")
# print(f"Average elas RMSE: {rmse_elas_total / count}")


: 

In [ ]:
y_pred_cat.shape
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

type(y_pred_cat)

In [ ]:
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

plt.scatter(range(len(y_full_pd.iloc[1].values)), y_full_pd.iloc[1].values,
            label="True", facecolors='none', edgecolors='blue', s=8)
plt.scatter(range(len(y_pred_unscaled[1])), y_pred_unscaled[1],
            label="Predicted", facecolors='none', edgecolors='orange', s=8)
# plt.plot(y_full_pd.iloc[1].values, label="True")
# plt.plot(y_pred_unscaled[1], label="Predicted")
plt.legend()
plt.title("y_predicted vs y_actual")
plt.xlabel("Site ID (coordinate)")
plt.ylabel("Spatial property")
plt.show()


In [ ]:
# hyperparam search

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'estimator__num_leaves': [20, 31, 40, 50],
    'estimator__max_depth': [-1, 5, 10, 20],
    'estimator__min_data_in_leaf': [10, 20, 30],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__n_estimators': [100, 500, 1000]}

# model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', verbosity=0))
model = MultiOutputRegressor(LGBMRegressor(objective='regression', verbosity=-1))

search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", (-search.best_score_)**0.5)



##### Spatial data (M)

In [ ]:
def plot_wafer_property(df, property_col, title):
    plt.figure(figsize=(9, 5))
    for rc_value, group in df.group_by("RC"):
        x = group["#Run"].to_list()
        y = group[property_col].to_list()
        plt.scatter(x, y, label=f'RC {rc_value}', s=20)
    plt.xlabel("#Run")
    plt.ylabel(property_col)
    plt.title(title)
    plt.legend()
    plt.show()

plot_wafer_property(wafer_df, "Wafer property summary 1", "Wafer Property 1")
plot_wafer_property(wafer_df, "Wafer property summary 2", "Wafer Property 2")


##### Import timeseries data (S)

In [ ]:
"""Load data and make parquet files out of it"""

should_we_save_parquet_files = False

def _save_df_as_parquet_file(df: pl.dataframe, saving_location: str):
    df.write_parquet(saving_location)

def remove_unchanging_cols_from_df_and_save(df: pl.dataframe, col_name: str, should_we_save_parquet_files: bool) -> None:
    for run_id in df[col_name].unique().to_list():
        df_per_run    = df.filter(pl.col(col_name) == run_id)
        constant_cols = [col for col in df_per_run.columns
                     if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
                        #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
                         df_per_run.select(pl.col(col).n_unique()).item() == 1]
        df_per_run_filtered = df_per_run.drop(constant_cols)
        saving_location = f"{parquet_subfolder}/run_{run_id}.parquet"
        if should_we_save_parquet_files:
            _save_df_as_parquet_file(df_per_run_filtered, saving_location)

remove_unchanging_cols_from_df_and_save(log_df, "#Run", should_we_save_parquet_files)

# =============
# before making funcrtion:

# if should_we_save_parquet_files:
#     for run_id in log_df["#Run"].unique().to_list():
#         df_per_run = log_df.filter(pl.col("#Run") == run_id)
#         zero_cols  = [col for col in df_per_run.columns
#                      if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
#                         #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
#                          df_per_run.select(pl.col(col).n_unique()).item() == 1]
#         df_per_run_filtered = df_per_run.drop(zero_cols)
#         df_per_run_filtered.write_parquet(f"{parquet_subfolder}/run_{run_id}.parquet")


##### Timeseries data (S)

In [ ]:
run_number   = 15
parquet_file = f"./ASM_data/3. marathon1/Logs/split_by_run/run_{run_number}.parquet"
df           = pl.read_parquet(parquet_file)
df_pd        = df.to_pandas()
df_numeric   = df.select(pl.col(pl.NUMERIC_DTYPES))
X_np         = df_numeric.to_numpy()
X_scaled     = StandardScaler().fit_transform(X_np)

num_cols_to_plot = len(df_pd.columns)
num_rows         = math.ceil(math.sqrt(num_cols_to_plot))
num_cols_grid    = math.ceil(num_cols_to_plot / num_rows)

axes = df_pd.plot(subplots=True, figsize=(14, 12), layout=(num_rows, num_cols_grid), sharex=True, legend=False)

if isinstance(axes, np.ndarray):
    axes_flat = axes.flatten()
else:
    axes_flat = [axes]

column_names = df_pd.columns.tolist()

for i, ax in enumerate(axes_flat):
    if i < num_cols_to_plot: # Only set title for actual plots
        ax.set_title(column_names[i], fontsize='xx-small')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

for i in range(num_cols_to_plot, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.suptitle(f'Run #{run_number} {log_file}', fontsize='large', y=1.02) # Adjust y to prevent overlap
plt.show()

In [ ]:
data_dimensionality = DimensionalityEstimator.estimate_dataset_dimensionality(df)
print(f"Recommended latent layer size: {data_dimensionality:.1f}")


In [ ]:
from sklearn.model_selection import train_test_split


X_train_np, X_test_np = train_test_split(X_scaled, test_size=0.2, random_state=42)
X_train               = torch.tensor(X_train_np, dtype=torch.float32)
X_test                = torch.tensor(X_test_np, dtype=torch.float32)
